# Bakery data profile report

This companion to [data_profile.ipynb](data_profile.ipynb) is for Team Cinnamon. It explains what each file contains, where records are missing, and what happens when the sources are joined. The tables refresh when you run the notebook. No model is fitted.

Run all cells from the project’s Jupyter environment. The first cell loads the public course files and the local inflation CSV using the original notebook’s loading code. See the [data fact sheet](../docs/data-fact-sheet.md) for source documentation and open decisions.

In [11]:
import io
import json
from contextlib import redirect_stdout
from pathlib import Path

import pandas as pd
from IPython.display import Markdown, display

profile_path = Path("data_profile.ipynb")
if not profile_path.is_file():
    profile_path = Path("1_DatasetCharacteristics") / profile_path

profile = json.loads(profile_path.read_text(encoding="utf-8"))
setup_cell = next(cell for cell in profile["cells"] if cell["cell_type"] == "code")
profile_scope = {}
with redirect_stdout(io.StringIO()):
    exec(compile("".join(setup_cell["source"]), str(profile_path), "exec"), profile_scope)

sales, weather, kiel, inflation = (
    pd.DataFrame(profile_scope["data"][name]).replace("", None)
    for name in ("sales", "weather", "kiel", "inflation")
)
for frame in (sales, weather, kiel):
    frame["Datum"] = pd.to_datetime(frame["Datum"])
sales = sales.astype({"Warengruppe": int, "Umsatz": float})
weather = weather.astype({column: float for column in weather if column != "Datum"})
kiel["KielerWoche"] = kiel["KielerWoche"].astype(int)
inflation = inflation.astype({column: float for column in inflation if column != "month"})

sales_days = pd.DatetimeIndex(sales["Datum"].unique()).sort_values()
sales_span = pd.date_range(sales_days.min(), sales_days.max())
weather_days = pd.DatetimeIndex(weather["Datum"])
weather_span = pd.date_range(weather_days.min(), weather_days.max())
missing_sales_days = sales_span.difference(sales_days)
missing_weather_days = sales_span.difference(weather_days)
def show(table):
    display(table.style.hide(axis="index"))

## The four sources describe different things

A sales row records turnover for one product group on one date. Weather has at most one observation per date. Kiel Week lists event days only, and inflation has one record per month. The key is the field or combination of fields that identifies a row.

In [12]:
sources = [
    ("Sales", sales, "Date and product group", ["Datum", "Warengruppe"]),
    ("Weather", weather, "Date", ["Datum"]),
    ("Kiel Week", kiel, "Event date", ["Datum"]),
    ("Inflation", inflation, "Month", ["month"]),
]
overview = []
for name, frame, key_name, key in sources:
    dates = frame[key[0]]
    date_label = "%Y-%m-%d" if key[0] == "Datum" else None
    start, end = dates.min(), dates.max()
    overview.append({
        "Source": name, "Rows": len(frame), "Columns": len(frame.columns),
        "Key": key_name, "Repeated keys": frame.duplicated(key).sum(),
        "First": start.strftime(date_label) if date_label else start,
        "Last": end.strftime(date_label) if date_label else end,
    })
show(pd.DataFrame(overview))

expected_id = sales["Datum"].dt.strftime("%y%m%d") + sales["Warengruppe"].astype(str)
display(Markdown(
    f"The sales ID also identifies each record. There are {sales['id'].duplicated().sum()} "
    f"repeated IDs and {(sales['id'] != expected_id).sum()} disagreements with the date and group."
))

Source,Rows,Columns,Key,Repeated keys,First,Last
Sales,9334,4,Date and product group,0,2013-07-01,2018-07-31
Weather,2601,5,Date,0,2012-01-01,2019-08-01
Kiel Week,72,2,Event date,0,2012-06-16,2019-06-30
Inflation,91,5,Month,0,2012-01,2019-07


The sales ID also identifies each record. There are 0 repeated IDs and 0 disagreements with the date and group.

## A blank value and a missing row tell us different things

A blank value means a row exists but one field is empty. A missing row means the file has no entry for that date or product group. For example, an absent sales row could reflect closure or an unavailable product. The file alone cannot tell us which.

In [13]:
group_names = {
    1: "Bread", 2: "Rolls", 3: "Croissants",
    4: "Confectionery", 5: "Cakes", 6: "Seasonal bread",
}
sales_keys = pd.MultiIndex.from_frame(sales[["Datum", "Warengruppe"]])
lag_keys = pd.MultiIndex.from_arrays([
    sales["Datum"] - pd.Timedelta(days=7), sales["Warengruppe"],
])
sales["has_previous_week"] = lag_keys.isin(sales_keys)

display(Markdown(
    f"Sales appear on **{len(sales_days):,} of {len(sales_span):,} calendar dates**. "
    f"The remaining **{len(missing_sales_days)} dates** have no sales rows. "
    f"There are **{sales['Umsatz'].isna().sum()} blank turnover values** in recorded rows."
))
sales_summary = sales.groupby("Warengruppe").agg(
    records=("Umsatz", "size"),
    minimum=("Umsatz", "min"),
    median=("Umsatz", "median"),
    maximum=("Umsatz", "max"),
    with_previous_week=("has_previous_week", "sum"),
)
show(pd.DataFrame({
    "Product group": sales_summary.index.map(group_names),
    "Recorded rows": sales_summary["records"],
    "Absent on dates with other sales": len(sales_days) - sales_summary["records"],
    "Lowest turnover": sales_summary["minimum"].round(2),
    "Median turnover": sales_summary["median"].round(2),
    "Highest turnover": sales_summary["maximum"].round(2),
    "Without a seven-day match": sales_summary["records"] - sales_summary["with_previous_week"],
}))
display(Markdown(
    "Seasonal bread has recorded sales in October through January. Its actual offer calendar "
    "would tell us which other dates should be eligible for a forecast."
))

Sales appear on **1,819 of 1,857 calendar dates**. The remaining **38 dates** have no sales rows. There are **0 blank turnover values** in recorded rows.

Product group,Recorded rows,Absent on dates with other sales,Lowest turnover,Median turnover,Highest turnover,Without a seven-day match
Bread,1819,0,23.110000,120.050000,416.790000,40
Rolls,1819,0,175.000000,370.830000,1203.430000,40
Croissants,1819,0,37.740000,142.810000,565.940000,40
Confectionery,1766,53,27.430000,79.990000,430.500000,87
Cakes,1819,0,121.490000,265.960000,1879.460000,40
Seasonal bread,292,1527,7.050000,62.910000,172.870000,43


Seasonal bread has recorded sales in October through January. Its actual offer calendar would tell us which other dates should be eligible for a forecast.

## Weather code blanks are common and vary with the weather

A missing weather code may mean no notable weather was reported. The source also records an explicit code 0, so the report counts blanks separately. These weather observations are a stand-in for the forecast that would be available to the bakery.

In [14]:
weather_labels = {
    "Bewoelkung": "Cloud cover (octas)", "Temperatur": "Mean temperature (°C)",
    "Windgeschwindigkeit": "Mean wind speed (reported m/s)", "Wettercode": "Weather code",
}
weather_summary = []
for column, label in weather_labels.items():
    values = weather[column]
    blank_dates = weather.loc[values.isna(), "Datum"]
    weather_summary.append({
        "Field": label, "Lowest": round(values.min(), 2), "Highest": round(values.max(), 2),
        "Blank weather records": len(blank_dates),
        "Sales rows on those dates": sales["Datum"].isin(blank_dates).sum(),
    })
show(pd.DataFrame(weather_summary))
display(Markdown(
    f"The weather file has **{len(weather_span.difference(weather_days))} dates with no row** "
    f"within its full span. In the sales period, **{len(missing_weather_days)}** are missing "
    f"({', '.join(missing_weather_days.strftime('%Y-%m-%d'))})."
))
blank_code = weather["Wettercode"].isna()
show(pd.DataFrame({
    "Weather code": ["Blank", "Recorded"],
    "Mean temperature (°C)": [
        round(weather.loc[mask, "Temperatur"].mean(), 2)
        for mask in (blank_code, ~blank_code)
    ],
    "June records": [(mask & weather["Datum"].dt.month.eq(6)).sum() for mask in (blank_code, ~blank_code)],
    "December records": [(mask & weather["Datum"].dt.month.eq(12)).sum() for mask in (blank_code, ~blank_code)],
}))

Field,Lowest,Highest,Blank weather records,Sales rows on those dates
Cloud cover (octas),0.000000,8.000000,10,54
Mean temperature (°C),-10.250000,32.670000,0,0
Mean wind speed (reported m/s),3.000000,35.000000,0,0
Weather code,0.000000,95.000000,669,2309


The weather file has **169 dates with no row** within its full span. In the sales period, **3** are missing (2016-12-11, 2017-10-04, 2017-10-05).

Weather code,Mean temperature (°C),June records,December records
Blank,15.440000,91,22
Recorded,10.940000,148,178


## The proposed joins keep every sales record

A left join starts from sales and adds information when the date or month matches. Each sales row should appear once afterward. Low match coverage is expected for Kiel Week because the source lists event days only. Both inflation lags are shown for comparison. A month appearing in the CSV does not establish when its figures were published.

In [15]:
sales_month = sales["Datum"].dt.to_period("M")
join_inputs = [
    ("Weather by date", sales["Datum"], weather["Datum"]),
    ("Kiel Week by date", sales["Datum"], kiel["Datum"]),
    ("Inflation one month earlier", sales_month - 1, pd.PeriodIndex(inflation["month"], freq="M")),
    ("Inflation two months earlier", sales_month - 2, pd.PeriodIndex(inflation["month"], freq="M")),
]
joins = []
for label, left, right in join_inputs:
    left_keys = pd.Index(left.unique())
    right_keys = pd.Index(right.unique())
    shared = len(left_keys.intersection(right_keys))
    merged = pd.DataFrame({"key": left}).merge(pd.DataFrame({"key": right}), how="left", on="key", indicator=True)
    joins.append({
        "Join": label, "Sales keys matched": f"{shared:,} / {len(left_keys):,}",
        "Source keys matched": f"{shared:,} / {len(right_keys):,}",
        "Sales rows matched": merged["_merge"].eq("both").sum(),
        "Rows after join": len(merged),
    })
show(pd.DataFrame(joins))

inflation_summary = []
for column, label in {
    "hicp_all_items": "All-items index (2015 = 100)",
    "hicp_bread_cereals": "Bread and cereals index (2015 = 100)",
    "inflation_all_items_pct": "All-items annual change (%)",
    "inflation_bread_cereals_pct": "Bread and cereals annual change (%)",
}.items():
    values = inflation[column]
    inflation_summary.append({
        "Inflation measure": label, "Lowest": values.min(), "Highest": values.max(),
        "Blank months": values.isna().sum(),
    })
show(pd.DataFrame(inflation_summary))

Join,Sales keys matched,Source keys matched,Sales rows matched,Rows after join
Weather by date,"1,816 / 1,819","1,816 / 2,601",9318,9334
Kiel Week by date,"45 / 1,819",45 / 72,223,9334
Inflation one month earlier,61 / 61,61 / 91,9334,9334
Inflation two months earlier,61 / 61,61 / 91,9334,9334


Inflation measure,Lowest,Highest,Blank months
All-items index (2015 = 100),95.700000,106.400000,0
Bread and cereals index (2015 = 100),93.200000,105.300000,0
All-items annual change (%),-0.510000,2.630000,12
Bread and cereals annual change (%),0.200000,3.430000,12


## The seven-day comparison has gaps

The baseline uses turnover from the same product group exactly seven days earlier. The table shows when that comparison is available within each agreed evaluation period. The validation period can use earlier recorded sales when they would already be known at prediction time.

In [16]:
periods = [
    ("Training", "2013-07-01", "2017-07-31"),
    ("Validation", "2017-08-01", "2018-07-31"),
]
coverage = []
for label, start, end in periods:
    rows = sales.loc[sales["Datum"].between(start, end)]
    coverage.append({
        "Period": label, "From": start, "Through": end, "Sales rows": len(rows),
        "Dates with sales": rows["Datum"].nunique(), "Calendar dates": len(pd.date_range(start, end)),
        "Rows without a seven-day match": (~rows["has_previous_week"]).sum(),
    })
show(pd.DataFrame(coverage))

Period,From,Through,Sales rows,Dates with sales,Calendar dates,Rows without a seven-day match
Training,2013-07-01,2017-07-31,7493,1462,1492,248
Validation,2017-08-01,2018-07-31,1841,357,365,42


The report covers recorded turnover. It cannot measure unit sales, unmet demand, branch prices or promotions. The holiday calendars and Kaggle test file still need to be checked. The [data fact sheet](../docs/data-fact-sheet.md) records the pending decisions on missing records, seasonal availability, inflation timing and baseline fallback.

## First five rows of the current dataset

The preview starts with the first five recorded sales rows and adds weather and Kiel Week by date. It shows inflation using each of the two month offsets explored above. These columns illustrate the available data while the team confirms the inflation timing rule.

In [17]:
preview = (
    sales[["id", "Datum", "Warengruppe", "Umsatz"]].head(100)
    .merge(weather, on="Datum", how="left", validate="many_to_one")
    .merge(kiel, on="Datum", how="left", validate="many_to_one")
)
preview["KielerWoche"] = preview["KielerWoche"].fillna(0).astype(int)
target_month = preview["Datum"].dt.to_period("M")
inflation_rates = inflation.set_index("month")["inflation_all_items_pct"]
for lag in (1, 2):
    preview[f"All-items inflation (%) · {lag} mo. earlier"] = (
        (target_month - lag).astype(str).map(inflation_rates)
    )
preview["Datum"] = preview["Datum"].dt.strftime("%Y-%m-%d")
preview.insert(3, "Product group", preview["Warengruppe"].map(group_names))
display(preview.style.hide(axis="index").format(precision=2, na_rep="—"))

id,Datum,Warengruppe,Product group,Umsatz,Bewoelkung,Temperatur,Windgeschwindigkeit,Wettercode,KielerWoche,All-items inflation (%) · 1 mo. earlier,All-items inflation (%) · 2 mo. earlier
1307011,2013-07-01,1,Bread,148.83,6.00,17.84,15.00,20.00,0,1.97,1.55
1307021,2013-07-02,1,Bread,159.79,3.00,17.31,10.00,—,0,1.97,1.55
1307031,2013-07-03,1,Bread,111.89,7.00,21.07,6.00,61.00,0,1.97,1.55
1307041,2013-07-04,1,Bread,168.86,7.00,18.85,7.00,20.00,0,1.97,1.55
1307051,2013-07-05,1,Bread,171.28,5.00,19.98,12.00,—,0,1.97,1.55
1307061,2013-07-06,1,Bread,174.55,0.00,18.96,8.00,—,0,1.97,1.55
1307071,2013-07-07,1,Bread,92.64,0.00,21.38,9.00,—,0,1.97,1.55
1307081,2013-07-08,1,Bread,135.50,0.00,22.73,10.00,5.00,0,1.97,1.55
1307091,2013-07-09,1,Bread,136.05,0.00,23.27,8.00,—,0,1.97,1.55
1307101,2013-07-10,1,Bread,135.13,2.00,19.69,13.00,5.00,0,1.97,1.55
